Das Modell mit optimierten Hyperparametern wird fine-getuned und dann gespeichert

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd
import os
!pip install evaluate
import evaluate
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive/Daten")

df = pd.read_csv("kandis_cleaned.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df.rename(columns={"body":"text"}, inplace=True)


In [ ]:
df_codiert = pd.read_excel("Excel_eigencodierung_final.xlsx")
df_codiert.rename(columns={"nummer":"index"}, inplace=True)
df_codiert.shape[0]

In [ ]:
df = df.reset_index()
df = pd.merge(df_codiert, df, on=["text","index"], how="outer")
print(df.shape[0])
df = df.dropna(subset=["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]).copy(deep=True)
df.shape[0]

In [ ]:
print(df[["D_0"]].value_counts())
print(df[["D_1"]].value_counts())
print(df[["D_2"]].value_counts())
print(df[["D_3"]].value_counts())
print(df[["D_4"]].value_counts())
print(df[["D_5"]].value_counts())
print(df[["D_6"]].value_counts())
print(df[["D_7"]].value_counts())

In [ ]:
train_df, test_df = train_test_split(df[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]], test_size=0.2, random_state=5460)
train_df, val_df = train_test_split(df[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]], test_size=0.25, random_state=5460)

In [ ]:
label_columns = ["D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]

In [ ]:
MODEL = "deepset/gbert-large"
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=8,
    problem_type="multi_label_classification"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
train_labels = train_df[label_columns].values.tolist()
val_labels = val_df[label_columns].values.tolist()

In [ ]:
# Tokenize the data
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.25).astype(int)                          # Sobald die Wahrscheinlichkeit einer Kategorie über 0,25 liegt, wird eine 1 vergeben (sonst 0)
    return {
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_score_macro": f1_score(labels, preds, average="macro"),
    }

train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)
val_encodings = tokenizer(
    val_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# Create Dataset objects
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels
})



In [ ]:
training_args = TrainingArguments(
    output_dir="./results",                # Directory to save model checkpoints and results
    num_train_epochs=6,                    # Number of training epochs
    per_device_train_batch_size=8,        # Batch size for training
    per_device_eval_batch_size=8,         # Batch size for evaluation
    learning_rate=4e-05,                 # Learning rate
    weight_decay=0.01,
    warmup_steps= 100,                     # Regularization to prevent overfitting
    eval_strategy="epoch",           # Evaluate the model after every epoch
    save_strategy="epoch",                 # Save model checkpoints after every epoch
    metric_for_best_model="f1_score_macro",      # Maximize the F1 score during training
    greater_is_better=True,                # Indicate that higher F1 scores are better
    load_best_model_at_end=True,           # Load the best model (based on F1) after training
    logging_strategy="steps",              # Enable logging at regular intervals
    logging_steps=10                       # Log every 10 steps (adjust as needed)
)


#Hier wird kein custom-Trainer benutzt, weil ja auch nicht gewichtet wird (ich habe dafür keine Möglichkeit bei einer multi-label-Klassifikation gefunden)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
results

In [ ]:
def predict_with_model(model, tokenizer, texts, device):
    """
    Use the fine-tuned model to predict probabilities for the given texts.
    """
    model.to(device)
    model.eval()  # Ensure the model is in evaluation mode

    # Tokenize the input texts
    encodings = tokenizer(
        texts, truncation=True, padding=True, max_length=512, return_tensors="pt"
    )
    encodings = {key: tensor.to(device) for key, tensor in encodings.items()}  

    # Get predictions
    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probabilities = torch.sigmoid(logits) 

    probabilities = probabilities.cpu().numpy()
    predictions = (probabilities >=0.25).astype(int)


    return probabilities, predictions

In [ ]:
trainer.save_model("best_trainer_multi")
model.save_pretrained("best_model_multi")

Im folgenden Teil wird qualitativ die Güte der Klassifikation betrachtet.

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
test_df = test_df[test_df["D_0"] != 1] #Die unpolitischen Posts erkennt das Modell sehr gut, weshalb hier die als politisch klassifizierten Posts untersucht werden

In [ ]:
test_texts = test_df["text"].tolist()
test_true_labels = test_df[label_columns].values.tolist()
probabilities, predicted_indices = predict_with_model(model, tokenizer, test_texts, device)

test_df["true_label"] = test_true_labels
test_df["predicted_label"] = predicted_indices.tolist()

mismatches_df = test_df[test_df["true_label"] != test_df["predicted_label"]]
matches_df = test_df[test_df["true_label"] == test_df["predicted_label"]]

print(f"Number of mismatches: {len(mismatches_df)} (out of {len(test_df)})")
# Die hohe Anzahl der Mismatches ist natürlich nicht wünschenswert, meistens ist aber nur ein label falsch (und 7 richtig),
# die Mismatch-Anzahl ist somit vielleicht ein bisschen irreführend

In [ ]:
mismatches_df["true_labels"] = mismatches_df[label_columns].values.tolist()

In [ ]:
###################################
# Classify a custom text input
###################################
# Define a sample text input (students can modify this)
sample_text, true_label = mismatches_df.sample(1).iloc[0][["text","true_labels"]]

# Tokenize the input text
sample_encoding = tokenizer(
    sample_text, truncation=True, padding=True, max_length=512, return_tensors="pt"
).to(device)


# Get predictions
model.eval
with torch.no_grad():
    outputs = model(**sample_encoding)
    logits = outputs.logits
    probabilities = torch.sigmoid(logits)  # Convert logits to probabilities

probabilities = probabilities.cpu().numpy()
predictions = (probabilities >=0.25).astype(int)



# Display the results
print(f"Input text: {sample_text}")
print(f"true labels: {true_label}")
print(f"Predicted label: {predictions}")
print(f"Probabilities {probabilities}")
print("Class probabilities:") #Hier kann nochmal genauer eingesehen werden, wie das "Abschneiden funktioniert"

In [ ]:
probabilities